In [ ]:
from collections import Counter
import math

def entropy(y):
    """Berechnet die Entropie einer Liste von Labels"""
    counts = Counter(y)
    total = len(y)
    return -sum((c/total) * math.log2(c/total) for c in counts.values())

def split_data(X, y, feature_idx, threshold):
    """Teilt Daten anhand eines Features und Schwellwerts"""
    left_mask = [x[feature_idx] <= threshold for x in X]
    right_mask = [not m for m in left_mask]
    return ([X[i] for i in range(len(X)) if left_mask[i]],
            [y[i] for i in range(len(y)) if left_mask[i]],
            [X[i] for i in range(len(X)) if right_mask[i]],
            [y[i] for i in range(len(y)) if right_mask[i]])

def best_split(X, y):
    """Findet den besten Split (Feature, Threshold)"""
    best_gain = -1
    best_feature, best_threshold = None, None
    n_features = len(X[0])
    
    for feature_idx in range(n_features):
        values = sorted(set(x[feature_idx] for x in X))
        for i in range(len(values) - 1):
            threshold = (values[i] + values[i+1]) / 2
            _, y_left, _, y_right = split_data(X, y, feature_idx, threshold)
            
            if len(y_left) == 0 or len(y_right) == 0:
                continue
            
            gain = entropy(y) - (len(y_left)/len(y) * entropy(y_left) + 
                                 len(y_right)/len(y) * entropy(y_right))
            
            if gain > best_gain:
                best_gain = gain
                best_feature = feature_idx
                best_threshold = threshold
    
    return best_feature, best_threshold

class DecisionTree:
    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None
    
    def _build_tree(self, X, y, depth=0):
        n_samples = len(y)
        n_classes = len(set(y))
        
        # Abbruchbedingungen
        if (depth >= self.max_depth or 
            n_samples < self.min_samples_split or 
            n_classes == 1):
            return Counter(y).most_common(1)[0][0]
        
        # Besten Split finden
        feature, threshold = best_split(X, y)
        if feature is None:
            return Counter(y).most_common(1)[0][0]
        
        # Daten splitten
        X_left, y_left, X_right, y_right = split_data(X, y, feature, threshold)
        
        # Rekursiv Teilbäume bauen
        return {
            'feature': feature,
            'threshold': threshold,
            'left': self._build_tree(X_left, y_left, depth + 1),
            'right': self._build_tree(X_right, y_right, depth + 1)
        }
    
    def fit(self, X, y):
        self.tree = self._build_tree(X, y)
        return self
    
    def _predict_one(self, x, node):
        if not isinstance(node, dict):
            return node
        if x[node['feature']] <= node['threshold']:
            return self._predict_one(x, node['left'])
        return self._predict_one(x, node['right'])
    
    def predict(self, X):
        return [self._predict_one(x, self.tree) for x in X]

# Beispiel: Iris-Datensatz (einfach, ohne externe Libs)
if __name__ == "__main__":
    # Mini-Datensatz: [SepalLength, SepalWidth] -> Species (0=setosa, 1=versicolor)
    X = [[5.1, 3.5], [4.9, 3.0], [7.0, 3.2], [6.4, 3.2], [5.5, 2.3], [6.5, 2.8]]
    y = [0, 0, 1, 1, 1, 1]
    
    # Modell trainieren
    dt = DecisionTree(max_depth=3)
    dt.fit(X, y)
    
    # Vorhersagen
    test = [[5.0, 3.4], [6.0, 3.0]]
    print("Vorhersagen:", dt.predict(test))  # [0, 1]
